# Extraction: model selection and prompt engineering

Two considerations:

1. **Which model can run to extract the required data?** Langchain uses `with_structured_output` forces a tool call, which not every model supports. Furthermore, choice of model and model provider used may also impact the quality of the output and the cost 
2. **Does the prompt get the right values?** Design for prompt engineering may affect the type of output generated

Ground truth comes from `01_value_exploration.ipynb`; scoring uses
`expectations/expected.yaml` via `src/evaluation.py`. Assumptions are at the end.

---

# Part 1: Which model can run this

## LLM and provider

The LLM used for extraction is `llama-3.1-8b-instant`, served through Groq, at
`temperature: 0`. All settings can be found in `config.yml`.

Why Groq:

- Free, no card - the constraint that excluded GPT-4 and Claude on cost, not merit
- Nothing downloaded - Ollama needs 2-5 GB of disk and RAM per model
- Supports Pydantic schemas via structured output - `langchain-huggingface` raises `NotImplementedError`
- Returns correct values - a local model returned `28400000000.0`, folding the unit into the value
- The trade: 100k tokens/day - which is why answers are committed to `results/` rather than regenerated

Temperature 0 is what makes prompt iteration measurable: the model takes the
most likely token each time, so the same pages yield the same figures, and a
changed result can be attributed to the prompt rather than to sampling.

---

# Part 2: Does the prompt work

The model is settled. The question now is whether the prompt extracts the right
values from the right pages.

In [ ]:
import json
import re
import sys
from pathlib import Path

import pypdf

sys.path.insert(0, str(Path.cwd().parent))

from src.config import load_config
from src.extraction.extractor import build_chain, page_variables
from src.extraction.prompts import load_prompt
from src.extraction.schemas import ExtractionResult
from src.ingestion.parser import extract_pages
from src.llm import get_chat_model

config = load_config(Path.cwd().parent / "config.yml")
from src.ingestion.download import ensure_pdf

pdf_path = ensure_pdf(config.pdf_url)

# Page bindings come from config.yml, not from the prompt text.
PAGE_VARS = page_variables(config)

model = get_chat_model(config)
print(f"provider={config.provider}  model={config.model}  temperature={config.temperature}")
print(f"pages={config.pages}")
print(f"field bindings={PAGE_VARS}")

## The input

pypdf extracts the four cited pages. Each is prefixed with a marker so the model
can be held to a specific page, and so extracted values can be traced back.

In [2]:
from src.ingestion.download import ensure_pdf

pdf_path = ensure_pdf(config.pdf_url)
page_text = extract_pages(pdf_path, config.pages)

print(f"{len(page_text):,} characters from pages {config.pages}\n")
for line in page_text.splitlines():
    if line.startswith("--- page"):
        print(line)

8,916 characters from pages [5, 6, 8, 20]

--- page 5 ---
--- page 6 ---
--- page 8 ---
--- page 20 ---


## Why pages are required, not merely convenient

"Corporate Income Tax" appears eight times across seven pages, carrying a
different value nearly every time. The cell below finds them all.

In [ ]:


reader = pypdf.PdfReader(str(pdf_path))
for page_no, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""
    for match in re.finditer(r"Corporate Income Tax", text):
        excerpt = " ".join(text[match.start():match.start() + 95].split())
        print(f"  p{page_no:2d}: {excerpt}")

Feeding the whole document would leave the model choosing among:

| Page | Value | What it actually is |
|---|---|---|
| 5 | **28.4** | Revised FY2023, prose - **the answer** |
| 8 | 28.38 | the same figure, table precision |
| 8 | 23.07 / 24.26 | FY2022 actual / FY2023 estimated |
| 9 | 27.2% | share of Operating Revenue |
| 16 | 28.03 | Estimated **FY2024** - a different year |
| 26 | 28,380 / 28,029 | the same figures in $million |
| 27 | 3.9% | share of GDP |

Every one is a plausible answer to "the amount of Corporate Income Tax". They
differ by year, by unit, and by whether they are an amount or a proportion -
and nothing in the number itself says which. Only the surrounding text
disambiguates, which is why the page citation does the work: it eliminates
seven wrong answers before the model reads anything.

This is also why a bare float is not a usable output. `28.38` and `28.03` are
both defensible-looking; only the page and quote reveal which was read.

## Why an LLM rather than a regex

The five fields sit in four different shapes:

| Field | Shape in the document |
|---|---|
| Corporate Income Tax | prose - `revised to $28.4 billion` |
| Year-on-year change | prose - `$4.1 billion (17.0%) higher` |
| Total top-ups | table total - `Total 20,352` |
| Tax list | narrative spread over two pages |
| Fiscal position | table row - `OVERALL FISCAL POSITION 1.72 (0.35) (3.57)` |

A regex handles one shape; five shapes means five patterns to write and
maintain. The model reads all four with one prompt.

The risk this buys is **mis-selection**, not invention. Every value is present in
the supplied text - the model is copying, not recalling - but it can copy the
wrong one. That is exactly what happened below.

In [3]:
# The line the model must read for the fiscal position. Five numbers, and only
# the header line elsewhere on the page says which is which.
for line in page_text.splitlines():
    if "OVERALL FISCAL POSITION" in line or "Corporate Income Tax 2" in line:
        print(line.strip())

Corporate Income Tax 23.07 24.26 28.38 23.0 17.0
OVERALL FISCAL POSITION 1.72 (0.35) (3.57)


## v1: the first prompt

The production code uses only `prompts/extraction.yaml`. The first version of
that prompt is not shipped; its result is recorded below from a development
run (llama-3.1-8b-instant, temperature 0) so the comparison stays visible
without keeping a superseded prompt in the submission.

**What v1 already did:**

- Named the page for each field ("from page 5", "from page 20")
- Explained the two-fiscal-year structure (section 1 = FY2023, section 2 = FY2024)
- Warned that terms recur with different values across pages
- Explained the conventions: parentheses mean negative, units differ by table,
  strip thousands separators
- Required a verbatim quote and page for every value
- Told the model to match table figures to columns by reading the header

That is not a naive prompt. It got three of five fields right, and the two it
missed were missed for a reason worth understanding.


### v1's recorded result

Three of five fields correct. The two failures, with the page each value was
actually read from:

| Field | v1 returned | Correct | What went wrong |
|---|---|---|---|
| total_top_ups | 24.3 billion, p.8 | 20,352 million, p.20 | took the first plausible match ("Top-ups... of $24.3 billion") and stopped - wrong year, wrong unit, wrong page |
| fiscal_position | -3.6, p.5 | -3.57, p.8 | the unscoped "prefer prose" rule outranked the page citation, so it read p.5's rounded prose figure |

The other three fields (corporate_income_tax 28.4/p.5, yoy 17.0/p.5, and the
seven tax names) were already right. v1 also sometimes failed schema
validation outright - its vaguer instructions left the model unsure of a
field's SHAPE, emitting "page" where TaxList requires "pages".

<details><summary>Full recorded v1 output</summary>

```json
{
  "corporate_income_tax": {"value": 28.4, "unit": "billion", "page": 5},
  "corporate_income_tax_yoy": {"value": 17.0, "page": 5},
  "total_top_ups": {"value": 24.3, "unit": "billion", "page": 8},
  "operating_revenue_taxes": {"names": ["Corporate Income Tax", "Other Taxes",
    "Vehicle Quota Premiums", "Personal Income Tax", "Assets Taxes",
    "Betting Taxes", "Goods and Services Tax"], "pages": [5, 5, 5, 6, 6, 6, 6]},
  "fiscal_position": {"value": -3.6, "unit": "billion", "page": 5}
}
```

</details>


### What changed in v2

Three edits. Each one fixes a specific failure above.

| # | Change | v1 said | v2 says | Fixes |
|---|---|---|---|---|
| 1 | Page citations declared **binding**, with the trap named | "from page 20" | "PAGE 20 ONLY... a value under a different marker is the WRONG value. Check the marker above your quote before answering." Plus: do not take the similarly-named row from another page. | top-ups |
| 2 | Prose preference **scoped** to the cited page | "prefer the prose" | "prefer the prose ON THE CITED PAGE... this preference never justifies leaving the cited page" | fiscal position |
| 3 | Instruction to read **to the end** | "list the taxes named" | "PAGES 5 AND 6 ONLY - this is the strictest constraint in this task... if a tax name appears only in a table on another page, it does not belong in this list." No expected count is stated: naming one would leak the answer and defeat the scorer's count check. | tax list |

The pattern: v1 stated the right pages but never said that a plausible value
elsewhere was *wrong*. The model treated the citations as preferences and
resolved them against other instructions - which is exactly what rule 2 did to
the fiscal position, since "prefer prose" and "read page 8" pointed in opposite
directions and nothing said which won.

**Wording is load-bearing.** While templating the page numbers, "Read both pages
to the end" became "Read those pages to the end" - and the tax list dropped from
13 back to 7, reproducibly at temperature 0. Restoring the emphatic phrasing
restored the 13. A one-word change moved a field from pass to fail.

In [6]:
prompt = load_prompt("extraction", Path.cwd().parent / "prompts")
rendered = prompt.format_messages(page_text="<PAGES>", target_year="Revised FY2023", **PAGE_VARS)

print(rendered[0].content[:1100])

You extract figures from Singapore government budget documents.

You will be given selected pages, each introduced by a marker such as
"--- page 5 ---". Every value you return must come from the text supplied.
Do not use prior knowledge of Singapore budgets, and do not calculate,
estimate, or infer any figure that is not written in the text.

PAGE CITATIONS ARE BINDING

Each field below names the page it must come from. That page is a hard
constraint, not a hint:

  - Read only the text under that page's marker.
  - A value that looks right but sits under a different marker is the WRONG
    value. Do not return it.
  - Before answering, check the marker above the text you are quoting and
    confirm it matches the page named for that field.

This matters because the document repeats terms across pages with different
values. "Top-ups to Endowment and Trust Funds" appears on one page as a
FY2023 figure in $billion and on another as a FY2024 total in $million. Only
the cited page is corre

## Summary: the five fields, question and answer

| # | Field (the question) | Answer | Page |
|---|---|---|---|
| 1 | Corporate Income Tax collections (Revised FY2023) | $28.4 billion | 5 |
| 2 | Year-on-year change in Corporate Income Tax | 17.0% | 5 |
| 3 | Total top-ups to Endowment and Trust Funds (Estimated FY2024) | $20,352 million | 20 |
| 4 | Taxes named in Operating Revenue | 7 names: Corporate Income Tax, Other Taxes, Vehicle Quota Premiums, Personal Income Tax, Assets Taxes, Betting Taxes, Goods and Services Tax | 5-6 |
| 5 | Overall Fiscal Position (Revised FY2023) | -$3.57 billion (a deficit) | 8 |

The full result, with each value's quote and provenance, is below.

In [7]:
chain = build_chain(model)
result = chain.invoke(
    {"page_text": page_text, "target_year": "Revised FY2023", **PAGE_VARS}
)
print(json.dumps(result.model_dump(), indent=2))

{
  "corporate_income_tax": {
    "quote": "Corporate Income Tax collections are revised to $ 28.4 billion, which is $4.1 billion (17.0%) higher than the Estimated FY2023 figure due to stronger-than-expected economic growth in 202 2.",
    "value": 28.4,
    "unit": "billion",
    "page": 5
  },
  "corporate_income_tax_yoy": {
    "quote": "Corporate Income Tax collections are revised to $ 28.4 billion, which is $4.1 billion (17.0%) higher than the Estimated FY2023 figure due to stronger-than-expected economic growth in 202 2.",
    "value": 17.0,
    "page": 5
  },
  "total_top_ups": {
    "quote": "Total 20,352",
    "value": 20352.0,
    "unit": "million",
    "page": 20
  },
  "operating_revenue_taxes": {
    "quote": "Revised FY2023 Operating Revenue is $104.3 billion, which is $7.6 billion (7.9%) higher than the Estimated FY2023 figure. This increase is mainly due to higher collections from Corporate Income Tax, Other Taxes, Vehicle Quota Premiums, Personal Income Tax, Assets Tax

## Scoring

Checked against the known values, including the page each was read from - a
right number from the wrong page would still be a failure of the instruction.

## Provenance

Every field carries the text it was read from. This is what makes the result
checkable rather than merely plausible - and it is how the v1 failures were
diagnosed, since the quotes named the wrong pages.

In [9]:
for field in ("corporate_income_tax", "total_top_ups", "fiscal_position"):
    cited = getattr(result, field)
    print(f"{field}  (page {cited.page})")
    print(f"    {cited.quote[:150]}\n")

corporate_income_tax  (page 5)
    Corporate Income Tax collections are revised to $ 28.4 billion, which is $4.1 billion (17.0%) higher than the Estimated FY2023 figure due to stronger-

total_top_ups  (page 20)
    Total 20,352

fiscal_position  (page 8)
    OVERALL FISCAL POSITION 1.72 (0.35) (3.57)



## Determinism

Temperature 0 means the model takes the most likely token at each step rather
than sampling, so the same pages return the same figures. For extraction that is
the whole point: a figure that changes between runs cannot be trusted, and a
disagreement between two runs gives no way to tell which was right.

The cell below runs the same extraction twice and compares.

In [11]:
runs = [
    chain.invoke({"page_text": page_text, "target_year": "Revised FY2023", **PAGE_VARS})
    for _ in range(2)
]

SCALAR_FIELDS = (
    "corporate_income_tax",
    "corporate_income_tax_yoy",
    "total_top_ups",
    "fiscal_position",
)

for field in SCALAR_FIELDS:
    values = [getattr(run, field).value for run in runs]
    print(f"{field:28s} {values}  {'stable' if values[0] == values[1] else 'DRIFT'}")

corporate_income_tax         [28.4, 28.4]  stable
corporate_income_tax_yoy     [17.0, 17.0]  stable
total_top_ups                [20352.0, 20352.0]  stable
fiscal_position              [-3.57, -3.57]  stable


## Conclusion

**All five fields extract correctly, each from the page cited for it.**

### What the failures taught

The interesting result is v1, not v2. All three of its failures had one cause:
**an instruction the model treated as a preference rather than a constraint.**
v1 named the right page for every field and still read the wrong one twice,
because nothing told it that a plausible value on another page was *wrong*
rather than merely second-best.

The fix was not more context or a better model. It was making the constraint
explicit, naming the specific trap the model had fallen into, and scoping the
competing rule ("prefer prose") so it could not override the constraint.

### What made the failures visible

Provenance. Every field carries the text it was read from, so v1's errors were
diagnosable in seconds - the quotes named pages 8 and 5 where 20 and 8 were
required. Without quotes, `24.32` and `-3.6` are plausible numbers that would
have needed manual checking against the source to catch.

This is why the schema carries page and quote per field rather than bare floats.

### Limits of this result

- **Known answers were available.** Scoring works here because the correct
  figures were established by reading the document first. A pipeline over
  unseen documents has no such check, and provenance becomes the only guard.
- **Restricted input does much of the work.** Feeding four cited pages rather
  than 37 removes most of the ambiguity before the model sees it. The prompt has
  not been tested against the full document.
- **Model-specific.** The prompt was tuned against `llama-3.1-8b-instant`. A
  weaker model may need the constraints stated more heavily still.
- **One document.** Page citations, unit conventions, and the two-fiscal-year
  structure are all specific to this publication.

## Assumptions

- **Only the cited pages are read** (5, 6, 8, 20). The same term recurs across
  the document with different values - Corporate Income Tax appears on seven
  pages - so narrowing the input removes ambiguity the model would otherwise
  have to resolve. The specifications supply the page citations, so retrieval is
  already solved.
- **Each field is read from the page cited for it,** not from wherever the
  figure is most precise. Page 5 says "$28.4 billion"; page 8's table says
  28.38. The citation decides, so 28.4 is correct here.
- **Target year is Revised FY2023.** The fields are labelled 2024 but cite pages
  5 and 8, which report FY2023. The page citations were taken as authoritative.
  True FY2024 figures are on pages 13-16 and differ - Corporate Income Tax is
  28.03 there, with a year-on-year change of -1.2% rather than +17.0%.
- **The value must match the cited page exactly, in the form written there.**
  Page 5 says "$28.4 billion", so the answer is 28.4 with unit `billion` - not
  28.38 (page 8's table precision), not 28,400,000,000 (the unit expanded into
  the number), and not 28,400 (converted to million). The figure and its unit
  are separate fields precisely so the two cannot be silently combined. Local
  models failed exactly here, returning 28400000000.0.
- **Units are recorded, not converted.** Page 20 states $million while the other
  pages state $billion. Each value carries its unit; normalising would hide a
  1000x error.
- **Page bindings live in `config.yml`,** not in the prompt text, so changing a
  page changes it in one place. A field bound to a page that is never extracted
  fails at config load rather than silently returning a wrong answer.
- **Temperature is 0** so the same pages yield the same figures.
- **The prompt is document-specific.** Page citations, the two-fiscal-year
  structure, and the top-ups trap are all particular to this publication.
- **The prompt was tuned against `llama-3.1-8b-instant`.** A weaker model may
  need the constraints stated more heavily still.